In [1]:
pip install google-play-scraper

Note: you may need to restart the kernel to use updated packages.


In [2]:
"""
=============================================================================
  Senti-Recommend Project — Google Play Store AI Tools Scraper
=============================================================================
  Author  : PGD Data Science Student
  Purpose : Scrape AI tool apps from Google Play Store — metadata + reviews
  Library : google-play-scraper  (pip install google-play-scraper)
  Output  : Two CSV files  →  ai_tools_metadata.csv  |  ai_tools_reviews.csv
=============================================================================

INSTALL DEPENDENCIES FIRST (run in your terminal / Jupyter cell):
    pip install google-play-scraper pandas tqdm colorama

HOW IT WORKS:
    1. Searches Play Store using ~60 AI-focused keyword queries
    2. Deduplicates app results by app-id
    3. Fetches full metadata for every unique app found
    4. Scrapes up to MAX_REVIEWS_PER_APP reviews per app
    5. Saves two tidy CSVs ready for your NLP / CF pipeline

CATEGORIES COVERED (aligned with your project scope):
    - Generative Text / Chatbots
    - Image Generation
    - AI Coding Assistants
    - Productivity AI
    - Marketing AI
=============================================================================
"""

# ── Standard library ────────────────────────────────────────────────────────
import time
import random
import logging
from datetime import datetime
from pathlib import Path

# ── Third-party ─────────────────────────────────────────────────────────────
import pandas as pd
from tqdm import tqdm

try:
    from google_play_scraper import app, search, reviews, Sort
    from google_play_scraper.exceptions import NotFoundError
except ImportError:
    raise SystemExit(
        "\n[ERROR] google-play-scraper is not installed.\n"
        "Run:  pip install google-play-scraper\n"
    )

try:
    from colorama import Fore, Style, init as colorama_init
    colorama_init(autoreset=True)
    GREEN  = Fore.GREEN
    YELLOW = Fore.YELLOW
    RED    = Fore.RED
    CYAN   = Fore.CYAN
    RESET  = Style.RESET_ALL
except ImportError:
    GREEN = YELLOW = RED = CYAN = RESET = ""


# ════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit these values to control scraper behaviour
# ════════════════════════════════════════════════════════════════════════════

SEARCH_RESULTS_PER_QUERY = 30     # How many apps each keyword search returns
MAX_REVIEWS_PER_APP      = 200    # Reviews to pull per app (set to 500+ for richer data)
MIN_RATINGS_THRESHOLD    = 10     # Skip apps with fewer ratings (likely fake/dead apps)
COUNTRY                  = "us"   # Play Store country code
LANGUAGE                 = "en"   # Review language
DELAY_BETWEEN_REQUESTS   = (1, 3) # Random sleep (seconds) to avoid rate-limiting
OUTPUT_DIR               = Path(".")  # Where to save CSVs (current folder)

# ── Logging ──────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)


# ════════════════════════════════════════════════════════════════════════════
#  SEARCH KEYWORDS — Grouped by your 5 project categories
# ════════════════════════════════════════════════════════════════════════════

SEARCH_QUERIES = {

    "Generative Text / Chatbots": [
        "AI chatbot assistant",
        "ChatGPT AI chat",
        "AI writing assistant",
        "AI text generator",
        "GPT chat",
        "Claude AI assistant",
        "Gemini AI",
        "AI story writer",
        "AI content generator",
        "conversational AI",
        "AI email writer",
        "AI paraphraser",
    ],

    "Image Generation": [
        "AI image generator",
        "text to image AI",
        "AI art generator",
        "AI photo creator",
        "AI drawing app",
        "stable diffusion app",
        "AI avatar generator",
        "AI photo enhancer",
        "AI image editor",
        "AI portrait generator",
    ],

    "AI Coding Assistants": [
        "AI coding assistant",
        "AI code generator",
        "AI code completion",
        "programming AI helper",
        "AI debugger app",
        "code explain AI",
        "AI developer tools",
    ],

    "Productivity AI": [
        "AI productivity app",
        "AI note taker",
        "AI meeting summarizer",
        "AI task manager",
        "AI transcription app",
        "AI voice assistant productivity",
        "AI calendar assistant",
        "AI document summarizer",
        "AI PDF reader",
        "AI study helper",
        "AI flashcard maker",
    ],

    "Marketing AI": [
        "AI marketing tools",
        "AI social media post generator",
        "AI caption writer",
        "AI ad copy generator",
        "AI SEO app",
        "AI business tools",
        "AI copywriting assistant",
        "AI product description generator",
    ],
}


# ════════════════════════════════════════════════════════════════════════════
#  STEP 1 — SEARCH & COLLECT UNIQUE APP IDs
# ════════════════════════════════════════════════════════════════════════════

def collect_app_ids(queries: dict, n_results: int, country: str) -> dict:
    """
    Run every keyword query and return a dict:
        { app_id : category_label }
    Deduplicates across queries automatically.
    """
    app_to_category: dict[str, str] = {}

    for category, keywords in queries.items():
        print(f"\n{CYAN}══ Searching category: {category} ══{RESET}")

        for keyword in keywords:
            try:
                results_list = search(
                    keyword,
                    lang=LANGUAGE,
                    country=country,
                    n_hits=n_results,
                )
                new_count = 0
                for item in results_list:
                    app_id = item.get("appId", "")
                    if app_id and app_id not in app_to_category:
                        app_to_category[app_id] = category
                        new_count += 1

                print(f"  {GREEN}✓{RESET} [{keyword}] — {new_count} new apps "
                      f"(total unique so far: {len(app_to_category)})")

            except Exception as exc:
                print(f"  {YELLOW}⚠ [{keyword}] search failed: {exc}{RESET}")

            # Polite delay
            time.sleep(random.uniform(*DELAY_BETWEEN_REQUESTS))

    return app_to_category


# ════════════════════════════════════════════════════════════════════════════
#  STEP 2 — FETCH FULL APP METADATA
# ════════════════════════════════════════════════════════════════════════════

def fetch_metadata(app_to_category: dict, country: str, min_ratings: int) -> list[dict]:
    """
    For every app_id, pull full metadata from Play Store.
    Returns a list of clean flat dicts.
    """
    metadata_rows = []

    print(f"\n{CYAN}══ Fetching metadata for {len(app_to_category)} apps ══{RESET}")

    for app_id, category in tqdm(app_to_category.items(), desc="Metadata", unit="app"):
        try:
            info = app(app_id, lang=LANGUAGE, country=country)

            # ── Pricing classification (maps to your Feature Engineering) ──
            price_raw = info.get("price", 0)
            if price_raw == 0:
                # Check if IAP (in-app purchases) exist
                offers_iap = bool(info.get("offersIAP", False))
                pricing_model = "Freemium" if offers_iap else "Free"
            else:
                pricing_model = "Paid"

            # ── Difficulty / Complexity proxy (use install count) ──
            installs = info.get("realInstalls", 0) or 0

            row = {
                # Identifiers
                "app_id"            : app_id,
                "app_name"          : info.get("title", ""),
                "developer"         : info.get("developer", ""),
                "category_scraped"  : info.get("genre", ""),
                "project_category"  : category,      # YOUR 5 categories

                # Ratings & Popularity
                "avg_rating"        : info.get("score", None),
                "total_ratings"     : info.get("ratings", 0),
                "total_reviews"     : info.get("reviews", 0),
                "installs"          : installs,

                # App Details
                "description"       : info.get("description", "")[:1000],  # truncate
                "summary"           : info.get("summary", ""),
                "content_rating"    : info.get("contentRating", ""),
                "android_version"   : info.get("androidVersion", ""),
                "app_version"       : info.get("version", ""),
                "updated_on"        : info.get("updated", ""),
                "released_on"       : info.get("released", ""),
                "size"              : info.get("size", ""),

                # Feature Engineering columns (your scope)
                "pricing_model"     : pricing_model,
                "price_usd"         : price_raw,
                "offers_iap"        : info.get("offersIAP", False),
                "free"              : info.get("free", True),

                # Integration / Platform
                "url"               : info.get("url", ""),
                "icon_url"          : info.get("icon", ""),
                "genre_id"          : info.get("genreId", ""),

                # Histogram for detailed sentiment baseline
                "ratings_1_star"    : (info.get("histogram") or [0,0,0,0,0])[0],
                "ratings_2_star"    : (info.get("histogram") or [0,0,0,0,0])[1],
                "ratings_3_star"    : (info.get("histogram") or [0,0,0,0,0])[2],
                "ratings_4_star"    : (info.get("histogram") or [0,0,0,0,0])[3],
                "ratings_5_star"    : (info.get("histogram") or [0,0,0,0,0])[4],

                "scraped_at"        : datetime.utcnow().isoformat(),
            }

            # Filter out low-activity apps
            if (row["total_ratings"] or 0) < min_ratings:
                continue

            metadata_rows.append(row)

        except NotFoundError:
            log.debug(f"App not found (removed from store): {app_id}")
        except Exception as exc:
            log.warning(f"Metadata error for {app_id}: {exc}")

        time.sleep(random.uniform(*DELAY_BETWEEN_REQUESTS))

    print(f"{GREEN}✓ Metadata collected for {len(metadata_rows)} valid apps{RESET}")
    return metadata_rows


# ════════════════════════════════════════════════════════════════════════════
#  STEP 3 — SCRAPE REVIEWS (the core NLP input for your project)
# ════════════════════════════════════════════════════════════════════════════

def fetch_reviews(app_ids: list[str], max_reviews: int, country: str) -> list[dict]:
    """
    Pull user reviews for each app.
    Fetches MOST RELEVANT first (best signal), then NEWEST for recency.
    Returns a flat list of review dicts.
    """
    all_reviews = []

    print(f"\n{CYAN}══ Fetching reviews for {len(app_ids)} apps ══{RESET}")

    for app_id in tqdm(app_ids, desc="Reviews", unit="app"):

        for sort_order, sort_label in [
            (Sort.MOST_RELEVANT, "relevant"),
            (Sort.NEWEST,        "newest"),
        ]:
            try:
                batch, _ = reviews(
                    app_id,
                    lang=LANGUAGE,
                    country=country,
                    sort=sort_order,
                    count=max_reviews // 2,  # split quota between both sorts
                    filter_score_with=None,  # get all star ratings
                )

                for r in batch:
                    row = {
                        # Link back to metadata table
                        "app_id"            : app_id,

                        # Review identifiers
                        "review_id"         : r.get("reviewId", ""),
                        "user_name"         : r.get("userName", ""),

                        # ── CORE NLP INPUT ──────────────────────────────────
                        "review_text"       : r.get("content", ""),
                        "review_title"      : "",  # Play Store rarely has titles
                        # ────────────────────────────────────────────────────

                        # Numerical signal
                        "star_rating"       : r.get("score", None),  # 1–5
                        "thumbs_up_count"   : r.get("thumbsUpCount", 0),

                        # Temporal
                        "review_date"       : r.get("at", ""),

                        # Developer engagement (useful feature)
                        "reply_text"        : r.get("replyContent", ""),
                        "reply_date"        : r.get("repliedAt", ""),

                        # App version at time of review
                        "app_version_review": r.get("reviewCreatedVersion", ""),

                        # Metadata
                        "sort_source"       : sort_label,
                        "scraped_at"        : datetime.utcnow().isoformat(),
                    }
                    all_reviews.append(row)

            except Exception as exc:
                log.warning(f"Reviews error ({sort_label}) for {app_id}: {exc}")

            time.sleep(random.uniform(*DELAY_BETWEEN_REQUESTS))

    # Remove duplicate reviews (same review_id from both sort passes)
    df_temp = pd.DataFrame(all_reviews)
    if not df_temp.empty and "review_id" in df_temp.columns:
        df_temp = df_temp.drop_duplicates(subset="review_id")
        all_reviews = df_temp.to_dict("records")

    print(f"{GREEN}✓ Total reviews collected: {len(all_reviews)}{RESET}")
    return all_reviews


# ════════════════════════════════════════════════════════════════════════════
#  STEP 4 — SAVE TO CSV
# ════════════════════════════════════════════════════════════════════════════

def save_csv(data: list[dict], filename: str, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    filepath = output_dir / filename
    df = pd.DataFrame(data)
    df.to_csv(filepath, index=False, encoding="utf-8-sig")
    print(f"{GREEN}✓ Saved → {filepath}  ({len(df):,} rows × {len(df.columns)} cols){RESET}")
    return filepath


# ════════════════════════════════════════════════════════════════════════════
#  MAIN PIPELINE
# ════════════════════════════════════════════════════════════════════════════

def main():
    print(f"""
{CYAN}╔══════════════════════════════════════════════════════════╗
║   Senti-Recommend — Google Play Store AI Scraper         ║
║   Collecting data for your PGD Final Project             ║
╚══════════════════════════════════════════════════════════╝{RESET}
Settings:
  • Results per keyword query : {SEARCH_RESULTS_PER_QUERY}
  • Max reviews per app       : {MAX_REVIEWS_PER_APP}
  • Minimum ratings threshold : {MIN_RATINGS_THRESHOLD}
  • Country / Language        : {COUNTRY} / {LANGUAGE}
  • Output directory          : {OUTPUT_DIR.resolve()}
""")

    # ── STEP 1: Search ───────────────────────────────────────────────────────
    app_to_category = collect_app_ids(
        queries   = SEARCH_QUERIES,
        n_results = SEARCH_RESULTS_PER_QUERY,
        country   = COUNTRY,
    )
    print(f"\n{GREEN}Total unique apps found: {len(app_to_category)}{RESET}")

    if not app_to_category:
        print(f"{RED}No apps found. Check your internet connection.{RESET}")
        return

    # ── STEP 2: Metadata ─────────────────────────────────────────────────────
    metadata = fetch_metadata(
        app_to_category = app_to_category,
        country         = COUNTRY,
        min_ratings     = MIN_RATINGS_THRESHOLD,
    )

    if not metadata:
        print(f"{RED}No metadata collected.{RESET}")
        return

    metadata_path = save_csv(metadata, "ai_tools_metadata.csv", OUTPUT_DIR)

    # ── STEP 3: Reviews ──────────────────────────────────────────────────────
    valid_app_ids = [row["app_id"] for row in metadata]

    all_reviews = fetch_reviews(
        app_ids    = valid_app_ids,
        max_reviews= MAX_REVIEWS_PER_APP,
        country    = COUNTRY,
    )

    if all_reviews:
        reviews_path = save_csv(all_reviews, "ai_tools_reviews.csv", OUTPUT_DIR)

    # ── SUMMARY ──────────────────────────────────────────────────────────────
    print(f"""
{CYAN}══════════════════════════════════════════════════════════
  SCRAPING COMPLETE — Summary
══════════════════════════════════════════════════════════{RESET}
  Apps in metadata CSV  : {len(metadata):,}
  Total reviews scraped : {len(all_reviews):,}

  Output files:
    📄 {OUTPUT_DIR}/ai_tools_metadata.csv
    📄 {OUTPUT_DIR}/ai_tools_reviews.csv

  Next steps for your project:
    1. Run VADER / HuggingFace sentiment on  review_text  column
    2. Use  star_rating  as the CF interaction signal
    3. Join on  app_id  to merge sentiment scores into metadata
    4. Build your hybrid weighted scoring formula
{CYAN}══════════════════════════════════════════════════════════{RESET}
""")


# ════════════════════════════════════════════════════════════════════════════
#  OPTIONAL UTILITIES — Handy standalone functions for your notebooks
# ════════════════════════════════════════════════════════════════════════════

def quick_search(keyword: str, n: int = 20) -> pd.DataFrame:
    """
    Quickly search a keyword and return a DataFrame.
    Usage in Jupyter:
        df = quick_search("AI chatbot", n=10)
        df.head()
    """
    results_list = search(keyword, lang=LANGUAGE, country=COUNTRY, n_hits=n)
    return pd.DataFrame(results_list)


def get_single_app(app_id: str) -> dict:
    """
    Fetch full metadata for a single known app.
    Usage:
        info = get_single_app("com.openai.chatgpt")
        print(info)
    """
    return app(app_id, lang=LANGUAGE, country=COUNTRY)


def get_single_app_reviews(app_id: str, n: int = 100) -> pd.DataFrame:
    """
    Fetch reviews for a single known app.
    Usage:
        df = get_single_app_reviews("com.openai.chatgpt", n=50)
        df[['userName','score','content']].head()
    """
    batch, _ = reviews(
        app_id,
        lang=LANGUAGE,
        country=COUNTRY,
        sort=Sort.MOST_RELEVANT,
        count=n,
    )
    return pd.DataFrame(batch)


# ════════════════════════════════════════════════════════════════════════════
#  KNOWN AI TOOL APP IDs — Direct fallback list
#  Use these if search results miss popular apps
# ════════════════════════════════════════════════════════════════════════════

KNOWN_AI_APP_IDS = {
    # Generative Text / Chat
    "com.openai.chatgpt"                : "Generative Text / Chatbots",
    "com.anthropic.claude"              : "Generative Text / Chatbots",
    "com.google.android.apps.bard"      : "Generative Text / Chatbots",
    "com.microsoft.copilot"             : "Generative Text / Chatbots",
    "com.perplexity.ai"                 : "Generative Text / Chatbots",
    "ai.character.app"                  : "Generative Text / Chatbots",
    "com.nova.gpt"                      : "Generative Text / Chatbots",
    "com.jasper.android"                : "Generative Text / Chatbots",
    "com.writesonic.android"            : "Generative Text / Chatbots",
    "io.typeface.android"               : "Generative Text / Chatbots",
    "com.cohesive.android"              : "Generative Text / Chatbots",
    "com.copy.ai"                       : "Generative Text / Chatbots",

    # Image Generation
    "com.midjourney.android"            : "Image Generation",
    "com.stability.stable.diffusion"    : "Image Generation",
    "com.adobe.firefly"                 : "Image Generation",
    "com.canva.editor"                  : "Image Generation",
    "net.wombo.app"                     : "Image Generation",
    "com.nightcafe.creator"             : "Image Generation",
    "com.starryai"                      : "Image Generation",
    "com.ai.photo.artguru"              : "Image Generation",
    "com.dreampress.ai.art"             : "Image Generation",
    "io.lensa.app"                      : "Image Generation",

    # AI Coding Assistants
    "com.github.android"                : "AI Coding Assistants",
    "com.tabnine.TabNine"               : "AI Coding Assistants",
    "com.sourcegraph.cody"              : "AI Coding Assistants",
    "io.codeium.windsurf"               : "AI Coding Assistants",
    "co.replit.app"                     : "AI Coding Assistants",
    "com.deco.ide"                      : "AI Coding Assistants",

    # Productivity AI
    "com.notion.id"                     : "Productivity AI",
    "com.grammarly.android.keyboard"    : "Productivity AI",
    "com.otter.android"                 : "Productivity AI",
    "com.fireflies.ai"                  : "Productivity AI",
    "com.mem.ai"                        : "Productivity AI",
    "com.reclaim.android"               : "Productivity AI",
    "com.todoist.android.Todoist"       : "Productivity AI",
    "com.microsoft.teams"               : "Productivity AI",
    "com.notta.android"                 : "Productivity AI",
    "com.krisp.android"                 : "Productivity AI",
    "com.rewind.android"                : "Productivity AI",

    # Marketing AI
    "com.hootsuite.droid"               : "Marketing AI",
    "com.buffer.app"                    : "Marketing AI",
    "com.hubspot.android"               : "Marketing AI",
    "com.predis.ai"                     : "Marketing AI",
    "com.ocoya.android"                 : "Marketing AI",
    "io.pencil.android"                 : "Marketing AI",
}


def scrape_known_apps_only():
    """
    Use this function if you want to skip the search phase and directly
    scrape only the pre-defined KNOWN_AI_APP_IDS dictionary above.
    Faster and more targeted.

    Usage in Jupyter / script:
        metadata, reviews_data = scrape_known_apps_only()
    """
    print(f"{CYAN}Scraping {len(KNOWN_AI_APP_IDS)} known AI apps directly...{RESET}")

    metadata = fetch_metadata(
        app_to_category = KNOWN_AI_APP_IDS,
        country         = COUNTRY,
        min_ratings     = 0,
    )
    save_csv(metadata, "ai_tools_metadata.csv", OUTPUT_DIR)

    valid_ids  = [row["app_id"] for row in metadata]
    all_reviews = fetch_reviews(valid_ids, MAX_REVIEWS_PER_APP, COUNTRY)
    save_csv(all_reviews, "ai_tools_reviews.csv", OUTPUT_DIR)

    return metadata, all_reviews


# ════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════╗
║   Senti-Recommend — Google Play Store AI Scraper         ║
║   Collecting data for your PGD Final Project             ║
╚══════════════════════════════════════════════════════════╝
Settings:
  • Results per keyword query : 30
  • Max reviews per app       : 200
  • Minimum ratings threshold : 10
  • Country / Language        : us / en
  • Output directory          : C:\Users\haris\DS\NED\Project PGD DSAI Spring 2026\sentirecommend\data\raw


══ Searching category: Generative Text / Chatbots ══
  ✓ [AI chatbot assistant] — 14 new apps (total unique so far: 14)
  ✓ [ChatGPT AI chat] — 13 new apps (total unique so far: 27)
  ✓ [AI writing assistant] — 27 new apps (total unique so far: 54)
  ✓ [AI text generator] — 15 new apps (total unique so far: 69)
  ✓ [GPT chat] — 8 new apps (total unique so far: 77)
  ✓ [Claude AI assistant] — 6 new apps (total unique so far: 83)
  ✓ [Gemini AI] — 6 new apps (total unique so far: 89)
  ✓

Metadata:   0%|                                                                               | 0/591 [00:00<?, ?app/s]C:\Users\haris\AppData\Local\Temp\ipykernel_2500\2532567704.py:267: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at"        : datetime.utcnow().isoformat(),
Metadata: 100%|█████████████████████████████████████████████████████████████████████| 591/591 [24:11<00:00,  2.46s/app]


✓ Metadata collected for 482 valid apps
✓ Saved → ai_tools_metadata.csv  (482 rows × 30 cols)

══ Fetching reviews for 482 apps ══


Reviews:   0%|                                                                                | 0/482 [00:00<?, ?app/s]C:\Users\haris\AppData\Local\Temp\ipykernel_2500\2532567704.py:347: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at"        : datetime.utcnow().isoformat(),
Reviews: 100%|██████████████████████████████████████████████████████████████████████| 482/482 [40:48<00:00,  5.08s/app]


✓ Total reviews collected: 69727
✓ Saved → ai_tools_reviews.csv  (69,727 rows × 13 cols)

══════════════════════════════════════════════════════════
  SCRAPING COMPLETE — Summary
══════════════════════════════════════════════════════════
  Apps in metadata CSV  : 482
  Total reviews scraped : 69,727

  Output files:
    📄 ./ai_tools_metadata.csv
    📄 ./ai_tools_reviews.csv

  Next steps for your project:
    1. Run VADER / HuggingFace sentiment on  review_text  column
    2. Use  star_rating  as the CF interaction signal
    3. Join on  app_id  to merge sentiment scores into metadata
    4. Build your hybrid weighted scoring formula
══════════════════════════════════════════════════════════

